# Validación de SQL

### Importación de librerías

In [1]:
import os
import mysql.connector
from dotenv import load_dotenv
import pandas as pd

### Rutas

In [2]:
contrasenia = '../.env'
preliminares = '../datos/datos_preliminares.csv'
limpios = '../datos/datos_limpios.csv'

### Conexión a la base de datos.

In [3]:
# Contraseña del proyecto
load_dotenv(contrasenia)

# Conexión con MySQL
conexion = mysql.connector.connect(host='mysql', port=3306, user='root',
                                   password=os.getenv('MYSQL_ROOT_PASSWORD'))

# Verificar la conexión
if conexion.is_connected():
    print('Conexión exitosa')

Conexión exitosa


### Comprobando la existencia de las bases de datos

In [4]:
# Consulta SQL de ejemplo
consulta = """
SELECT *
FROM architecture.architect
LIMIT 1;
"""

# Ejecutar la consulta
prueba = pd.read_sql_query(consulta, conexion)

# Mostrar resultados
display(prueba)

/tmp/ipykernel_22598/1280364445.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  prueba = pd.read_sql_query(consulta, conexion)


,id,name,nationality,gender


### Datos a usar

In [5]:
df = pd.read_csv(preliminares)
df.head(2)

,db_id,pregunta,sql
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER..."


### Categorizando consultas MySQL 8.4.10

In [6]:
filtro = []

for _, fila in df.iterrows():

    try:
        # Seleccionar la base de datos correspondiente
        conexion.database = fila['db_id']

        # Ejecutar la consulta SQL
        with conexion.cursor() as cursor:
            cursor.execute(fila['sql'])
            cursor.fetchall()

        filtro.append(True)

    except mysql.connector.Error as error:
        filtro.append(False)

# Agregar resultados al DataFrame
df['valida'] = filtro

### Cantidad de útiles y no útiles

In [7]:
df['valida'].value_counts()

valida
True     4743
False    4416
Name: count, dtype: int64

### Filtrando únicamente los valores útiles

In [8]:
df_filtrado = df[filtro]

### Exportando datos útiles

In [9]:
df_filtrado.to_csv(limpios, index=False)